In [32]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [33]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")


In [34]:
#estraggo tutte le label  
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto 
myth_terms = list(set(labels + aliases)) #unisco le label e gli alias, tolgo i duplicati con set e riconverto in lista
len(myth_terms)

4455

In [35]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [36]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2258640


In [37]:
#query pr estrarre i dati dei beni storico artistici i batch 

query2 = f"""
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item (GROUP_CONCAT(?label; separator="|") AS ?labels) (GROUP_CONCAT(?title; separator="|") AS ?titles) (GROUP_CONCAT(?date; separator="|") AS ?dates) (GROUP_CONCAT(?subject; separator="|") AS ?subjects)
WHERE {{
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .

  OPTIONAL {{ ?item rdfs:label ?label }}
  OPTIONAL {{ ?item dc:title ?title }}
  OPTIONAL {{ ?item dc:date ?date }}
  OPTIONAL {{ ?item a-cd:subject ?subject }}
}}

GROUP BY ?item
LIMIT 1000
OFFSET 0

"""

sparql_arco.setQuery(query2)
results = sparql_arco.query().convert()


In [38]:
rows = []

for r in results["results"]["bindings"]:

    rows.append({
        "item": r["item"]["value"] if "item" in r else None,
        "label": r["labels"]["value"] if "labels" in r else None,
        "title": r["titles"]["value"] if "titles" in r else None,
        "date": r["dates"]["value"] if "dates" in r else None,
        "subject": r["subjects"]["value"] if "subjects" in r else None
        #"description": r["description"]["value"] if "description" in r else None,
        #"historical": r["historical"]["value"] if "historical" in r else None
    })

df_arco = pd.DataFrame(rows)

# Deduplicazione intelligente: per ogni item, mantieni la prima riga che ha label O title
# Prima sortaggio per prioritizzare righe con almeno uno tra label e title non-null
df_arco["has_text"] = df_arco["label"].notna() | df_arco["title"].notna()
df_arco = df_arco.sort_values(by=['item', 'has_text'], ascending=[True, False]).drop_duplicates(subset=['item'], keep='first')
df_arco = df_arco.drop(columns=['has_text'])

# Crea search_text unendo label e title
df_arco["search_text"] = (
    df_arco["label"].fillna("") + " " + df_arco["title"].fillna("")
)

print(f"Righe in df_arco dopo deduplica: {len(df_arco)}")
df_arco[[ "subject"]].head(10)

Righe in df_arco dopo deduplica: 1000


,subject
600,
173,
474,
910,
111,
25,
846,
835,
810,
20,


In [39]:
# Creiamo search_text solo in subject  

import re

# Pre-normalizzazione dei termini (fondamentale)
myth_terms_clean = [
    t.strip()
    for t in myth_terms
    if isinstance(t, str) and t.strip() != ""
]

# Funzione robusta di matching
def find_matched_terms(text, terms_list):
    if not isinstance(text, str):
        return None

    matched = []

    for term in terms_list:
        if re.search(r'\b' + re.escape(term) + r'\b', text):
            matched.append(term)
    
    return ", ".join(matched) if matched else None


# Applica il matching in modo sicuro
df_arco["matched_term"] = df_arco["subject"].apply(
    lambda text: find_matched_terms(text, myth_terms_clean)
)

# Filtra solo le righe con almeno un match
df_matches = df_arco[df_arco["matched_term"].notna()].copy()

print(f"Righe con almeno 1 match: {len(df_matches)}")

Righe con almeno 1 match: 25


In [40]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 1000
Beni con match mitologico: 25


In [41]:
#vediamo i risultati
df_matches

,item,label,title,date,subject,search_text,matched_term
995,https://w3id.org/arco/resource/HistoricOrArtis...,Diana davanti al consesso degli dei chiede ver...,Diana davanti al consesso degli dei chiede ver...,ca 1619-ca 1619|ca 1619-ca 1619,Diana chiede a Giove di donarle l'eterna vergi...,Diana davanti al consesso degli dei chiede ver...,"Diana, Giove"
633,https://w3id.org/arco/resource/HistoricOrArtis...,"fuga in Egitto (dipinto, elemento d'insieme) b...",dipinto (elemento d'insieme)|dipinto (elemento...,1777-1778|1777-1778,fuga in Egitto|fuga in Egitto,"fuga in Egitto (dipinto, elemento d'insieme) b...",Egitto
549,https://w3id.org/arco/resource/HistoricOrArtis...,"Mercurio decapita Argo addormentato (dipinto, ...",dipinto (ciclo)|dipinto (ciclo),ca 1577-ante 1586|ca 1577-ante 1586,Mercurio decapita Argo addormentato|Mercurio d...,"Mercurio decapita Argo addormentato (dipinto, ...","Argo, Mercurio"
944,https://w3id.org/arco/resource/HistoricOrArtis...,Madonna della Misericordia tra san Domenico e ...,dipinto (elemento d'insieme)|dipinto (elemento...,1541-1541|1541-1541,Madonna della Misericordia tra san Domenico e ...,Madonna della Misericordia tra san Domenico e ...,Misericordia
452,https://w3id.org/arco/resource/HistoricOrArtis...,Natura morta con ottavino (stampa) di Cordio N...,stampa|stampa,1947-1947|1947-1947,Natura morta con ottavino|Natura morta con ott...,Natura morta con ottavino (stampa) di Cordio N...,Natura
267,https://w3id.org/arco/resource/HistoricOrArtis...,"Saturno (statua, elemento d'insieme) - ambito ...",statua (elemento d'insieme)|statua (elemento d...,post 1700-ante 1799|post 1700-ante 1799,Saturno|Saturno,"Saturno (statua, elemento d'insieme) - ambito ...",Saturno
926,https://w3id.org/arco/resource/HistoricOrArtis...,"Amorino, Cupido (statuetta, pendant) - ambito ...","Amorino, Cupido (statuetta, pendant)|Amorino, ...",1800-1850|1800-1850,Cupido|Cupido,"Amorino, Cupido (statuetta, pendant) - ambito ...",Cupido
180,https://w3id.org/arco/resource/HistoricOrArtis...,"Clizia (statua, opera isolata) by Parodi Filip...",statua (opera isolata)|statua (opera isolata),post 1680-ante 1683|post 1680-ante 1683,Clizia|Clizia,"Clizia (statua, opera isolata) by Parodi Filip...",Clizia
691,https://w3id.org/arco/resource/HistoricOrArtis...,"Diana e fauno (stampa) di Cambiaso Luca, Piagg...",stampa (di traduzione)|stampa (di traduzione),ca 1770-ca 1810|ca 1770-ca 1810,Diana e fauno|Diana e fauno,"Diana e fauno (stampa) di Cambiaso Luca, Piagg...",Diana
921,https://w3id.org/arco/resource/HistoricOrArtis...,Tempo che scopre la Verità (dipinto) di Vaccar...,dipinto|dipinto,1690-post 1699|1690-post 1699,Tempo che scopre la Verità|Tempo che scopre la...,Tempo che scopre la Verità (dipinto) di Vaccar...,Verità
